# Gemini Drive notes → n8n HQ Tasks

Runtime → **Run all**. Sign in with a Save 5 Hours Google account (`@save5hours.ch`, `antubejar96@gmail.com`, `deevlylabs@gmail.com`, or `roman.cajka@gmail.com`).

This creates a real Google Doc and POSTs `{fileId, text, googleAccessToken}` to n8n. You do **not** need an n8n login or `WEBHOOK_SECRET`.

Keep `VERIFY_NOTES` in sync with `fixtures/drive-verify-notes.txt`.

In [ ]:
%pip install -q google-api-python-client google-auth-httplib2 google-auth-oauthlib requests

from google.colab import auth
auth.authenticate_user()

import google.auth
from googleapiclient.discovery import build
import requests

WEBHOOK = "https://n8n-production-192e.up.railway.app/webhook/meeting-notes-drive"
TITLE = "Gemini notes — Drive path verification (n8n)"
VERIFY_NOTES = (
    "Gemini notes — Drive path verification (n8n)\n\n"
    "Attendees: Antoine Bejarano Alvarez, Martin, Roman Cajka.\n\n"
    "Actions agreed:\n"
    "- Antoine will publish the Drive webhook runbook in HQ this week.\n"
    "- Martin will review HQ Tasks with Origin Meeting after the Drive file lands.\n"
    "- Roman will confirm the Meet Recordings folder URL on the Drive confirmation task.\n"
)

creds, _ = google.auth.default()
token = getattr(creds, "token", None) or ""
if not token:
    raise SystemExit("No Google access token. Re-run after signing in.")

docs = build("docs", "v1", credentials=creds)
doc = docs.documents().create(body={"title": TITLE}).execute()
file_id = doc["documentId"]
docs.documents().batchUpdate(
    documentId=file_id,
    body={"requests": [{"insertText": {"location": {"index": 1}, "text": VERIFY_NOTES}}]},
).execute()

resp = requests.post(
    WEBHOOK,
    json={
        "fileId": file_id,
        "name": TITLE,
        "mimeType": "application/vnd.google-apps.document",
        "webViewLink": f"https://docs.google.com/document/d/{file_id}/edit",
        "text": VERIFY_NOTES,
        "googleAccessToken": token,
    },
    timeout=120,
)
print("HTTP", resp.status_code)
print("FILE_ID", file_id)
print("FILE_URL", f"https://docs.google.com/document/d/{file_id}/edit")
print(resp.text[:500])
resp.raise_for_status()